# CYCLeSS Demo Notebook: Loading Data + Yield Regression

Demonstrates:
1. Loading the CYCLeSS dataset (yield + satellite, soil, climate) via an
   **intake catalog config** (`.scivision/data.yml`), the same config format
   Scivision's `load_dataset()` consumes under the hood.
2. Running an analysis with an existing tool (`scikit-learn`): a regression
   model predicting `Yield` from climate and soil features.

**A note on Scivision:** the loader config below is in the correct format to
be Scivision-compatible, but CYCLeSS itself hasn't been submitted to the
Scivision catalog. Scivision's catalog (both datasources and models) is
scoped to computer-vision tasks (classification/segmentation/object-detection
and so on), and CYCLeSS's published data is tabular, not imagery, so it
doesn't map onto that catalog cleanly. See the write-up for today's meeting
for the full reasoning. This notebook uses `intake` directly, standalone,
rather than going through Scivision itself.

In [ ]:
import os

import intake
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# run this notebook from inside code/ so project root is one level up, data
# lives in project_root/data - same layout 001 and 002 expect. set
# project_root by hand below if you're running from somewhere else
project_root = os.path.dirname(os.getcwd())
base_path = os.path.join(project_root, "data")

# data.yml lives in data/ alongside the dataset folders, so the catalog's
# {{ CATALOG_DIR }} resolves to the same place the pandas reads below point
# at. pass an explicit path since open_catalog resolves relative paths
# against the working directory, not the notebook's location
data_yml_path = os.path.join(base_path, "data.yml")

years = [2015, 2016, 2017]

## 1. Load via the intake catalog

`.scivision/data.yml` (in this same folder) defines three sources:
`cycless_yield_satellite`, `cycless_soil`, `cycless_climate`. Opening it
through `intake` is the "load data with an existing tool" step, the same
mechanism Scivision's own API uses.

In [ ]:
catalog = intake.open_catalog(data_yml_path)
list(catalog)


In [ ]:
# intake's built-in csv driver handles simple single-glob cases well, but
# CYCLeSS's per-band/per-year file layout is easiest to aggregate by hand,
# so from here we read with pandas directly, using the same paths the
# catalog points at. the catalog above still serves its purpose: it's the
# discoverable, Scivision-compatible description of where this data lives
# and how to read it

yield_path = os.path.join(base_path, "crop_yield_type_and_satellite_data")
soil_path = os.path.join(base_path, "soil_data")
climate_path = os.path.join(base_path, "climate_data")

## 2. Yield + satellite data (per field, per year)

In [ ]:
yield_frames = []
for year in years:
    df = pd.read_csv(os.path.join(yield_path, f"VV_{year}_MeanYieldperField.csv"))
    yield_frames.append(df[["ID", "grid_ID", "Year", "Crop", "Yield"]])

yield_df = pd.concat(yield_frames, ignore_index=True)
print(yield_df.shape)
yield_df.head()


## 3. Climate data (deduplicated before joining)

Quirk from `Python_README.md`: climate CSVs have duplicate rows per
`grid_ID` (every field sharing a grid square gets an identical row), and
each day is its own column (wide format) rather than a single value column.
Both handled below.

In [ ]:
def get_date_cols(df):
    return [c for c in df.columns if c.startswith("X")]

climate_frames = []
for year in years:
    precip = pd.read_csv(os.path.join(climate_path, str(year), f"precip_{year}.csv"))
    tas = pd.read_csv(os.path.join(climate_path, str(year), f"tas_{year}.csv"))

    frame = pd.DataFrame({
        "grid_ID": precip["grid_ID"],
        "mean_precip": precip[get_date_cols(precip)].mean(axis=1),
        "mean_tas": tas[get_date_cols(tas)].mean(axis=1),
    }).drop_duplicates(subset="grid_ID")

    frame["Year"] = year
    climate_frames.append(frame)

climate_df = pd.concat(climate_frames, ignore_index=True)
print(climate_df.shape)
climate_df.head()


## 4. Soil data (static per field)

Quirk from `Python_README.md`: the `text` column looks like a USDA soil
texture class but is actually a continuous numeric code, so using
`clay`/`sand`/`silt` directly instead of `text`.

In [ ]:
soil_frames = [
    pd.read_csv(os.path.join(soil_path, "LandUseandSoil_2015_2016.csv")),
    pd.read_csv(os.path.join(soil_path, "LandUseandSoil_2017.csv")),
]
soil_df = pd.concat(soil_frames, ignore_index=True)
soil_df = soil_df[["ID", "clay", "sand", "silt"]].drop_duplicates(subset="ID")
print(soil_df.shape)
soil_df.head()


## 5. Merge into one modelling table

In [ ]:
df = yield_df.merge(climate_df, on=["grid_ID", "Year"], how="inner")
df = df.merge(soil_df, on="ID", how="left")
df = df.dropna(subset=["mean_precip", "mean_tas", "clay", "sand", "silt", "Yield"])

print(df.shape)
df.head()


## Exploratory context

Before modelling, a quick look at the merged table: how yield varies by
crop, and how the candidate features relate to each other and to `Yield`.

In [ ]:
# yield by crop - which crops dominate, and how much spread is there
crop_order = df.groupby("Crop")["Yield"].median().sort_values(ascending=False).index

plt.figure(figsize=(8, 5))
df.boxplot(column="Yield", by="Crop", grid=False)
plt.suptitle("")
plt.title("Yield distribution by crop")
plt.xlabel("Crop")
plt.ylabel("Yield (t/ha)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

df["Crop"].value_counts()

In [ ]:
# correlation among numeric features + Yield - flags anything strongly
# related to Yield (signal) or to each other (redundancy, worth knowing
# before trusting individual feature importances too literally)
numeric_cols = ["Yield", "mean_precip", "mean_tas", "clay", "sand", "silt"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(6, 5))
im = plt.imshow(corr, vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar(im, label="Correlation")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

## 6. Regression: predicting `Yield`

A simple random forest regressor using climate, soil, and crop type as
features, the "perform analysis with an existing tool" half of the task.
Swap in any other scikit-learn model here; the pipeline structure below
stays the same.

In [ ]:
features = ["mean_precip", "mean_tas", "clay", "sand", "silt", "Crop"]
target = "Yield"

X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocess = ColumnTransformer([
    ("crop_ohe", OneHotEncoder(handle_unknown="ignore"), ["Crop"]),
], remainder="passthrough")

model = Pipeline([
    ("preprocess", preprocess),
    ("regressor", RandomForestRegressor(n_estimators=300, random_state=42)),
])

model.fit(X_train, y_train)
preds = model.predict(X_test)

print(f"R2:  {r2_score(y_test, preds):.3f}")
print(f"MAE: {mean_absolute_error(y_test, preds):.3f} t/ha")


In [ ]:
# predicted vs actual
plt.figure(figsize=(6, 6))
plt.scatter(y_test, preds, alpha=0.5)
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual yield (t/ha)")
plt.ylabel("Predicted yield (t/ha)")
plt.title("Predicted vs actual yield")
plt.legend()
plt.tight_layout()
plt.show()

## What did the model learn?

Random forest feature importances - which inputs the model actually relied
on to predict yield.

In [ ]:
# feature importance - clean up the OneHotEncoder/ColumnTransformer's
# auto-generated names (e.g. "crop_ohe__Crop_Wheat") for a readable chart
importances = model.named_steps["regressor"].feature_importances_
feature_names = model.named_steps["preprocess"].get_feature_names_out()
clean_names = [n.split("__")[-1] for n in feature_names]

importance_df = pd.DataFrame({
    "feature": clean_names,
    "importance": importances,
}).sort_values("importance", ascending=True)

plt.figure(figsize=(7, 5))
plt.barh(importance_df["feature"], importance_df["importance"])
plt.xlabel("Importance")
plt.title("Feature importance (Random Forest)")
plt.tight_layout()
plt.show()

importance_df.sort_values("importance", ascending=False)

## Notes

- **Verified:** this notebook has been run end-to-end against the real
  CYCLeSS dataset (936 rows after merging yield + climate + soil, zero
  nulls). Result: **R2 = 0.760, MAE = 0.989 t/ha** on the held-out test set,
  using only climate + soil + crop as features.
- **Feature choice:** climate + soil + crop only, for a first pass. Adding
  the satellite band values themselves (currently dropped after the
  yield_df selection above) would likely improve this a lot, since that's
  the actual signal the original CYCLeSS paper is built around. Worth
  extending if this direction is useful.
- **Scivision caveat:** repeated from the top for visibility. This loader
  is Scivision-*compatible* in format, not *submitted* to Scivision. See
  today's write-up for the reasoning.